# MoLFormer-XL — DIMER E2E molecular-property classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/molformer-chemistry-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/molformer-chemistry-pipeline/blob/main/tutorials/molformer_chemistry_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ibm--research%2FMoLFormer--XL--both--10pct-ffcc4d?style=flat)](https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct) [![Upstream](https://img.shields.io/badge/Upstream-IBM%2Fmolformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/IBM/molformer) [![arXiv](https://img.shields.io/badge/arXiv-2106.09553-b31b1b.svg)](https://arxiv.org/abs/2106.09553)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** SMILES embeddings and bounded molecular-property classification fine-tuning

**This notebook is standalone.** It carries the repository's package (3 modules under `src/molformer_chemistry_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `361063d0ad524ef77cf39b08469f6be770dc550f` (~187 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned MoLFormer-XL snapshot (7 files, ~187 MB, including the two Python files the loader executes), loads the tokenizer without remote code and the model with it, generates a deterministic 64-molecule constitutional-isomer dataset in code (no download), validates the SMILES and the dataset contract, splits it into stratified train/validation/test sets, shows how SMILES are tokenized, computes mean-pooled molecule embeddings, measures a majority-class and a molecular-formula baseline on the test split, runs a bounded AdamW fine-tuning of the property-classification head and the last two encoder layers, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six freshly generated molecules, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes well under a minute of model time.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled molecules as a CSV (`id,smiles,label`), a JSON array or a JSONL file. They pass through the same SMILES validation, stratified split, baselines, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the synthetic sample. The expected schema, the accepted SMILES characters and the token ceiling are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

MoLFormer-XL is a chemical language model: it reads molecules written as **SMILES strings** and was pretrained with a masked-token objective on a very large corpus of them (the upstream card describes 1.1 billion molecules from PubChem and ZINC; this checkpoint is the variant trained on a 10 % sample of both). Its notable architectural choice is **linear attention with random feature maps**, which is why it scales to that corpus — and why it behaves differently from an ordinary transformer in one respect that matters for reproducibility, explained in Section 3.

This tutorial adapts it to a molecular-property task. The tutorial dataset is synthetic but chemically literal: every molecule is paired with a **constitutional isomer** of itself — the same molecular formula, the same atoms, a different skeleton. One member is an unbranched chain (`CCCCCCCO`); the other carries one of those carbons as a methyl branch (`CCCC(C)CCO`). A classifier that only knows the molecular formula therefore cannot do better than chance, by construction, which is what makes the fine-tuned model's result worth reading.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify an immutable snapshot **including the Python the loader executes**; understand what `trust_remote_code=True` buys and costs here, and why this package overrides `deterministic_eval`; validate SMILES syntactically and split a labelled dataset without leakage; read how SMILES become tokens; extract mean-pooled molecule embeddings; measure majority-class and molecular-formula baselines; run a bounded fine-tuning with explicit hyperparameters; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify new molecules; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** regression on continuous properties, the published MoleculeNet benchmarks, molecule generation or optimisation, 3D conformers or descriptors, chemical validity checking (no RDKit is installed), and the other MoLFormer checkpoints. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — the default fine-tuning is a couple of seconds — and CUDA is used automatically when present.
- **Knowledge:** how a molecule is written as SMILES, what a constitutional isomer is, and how accuracy, macro-F1 and AUROC differ.
- **Remote code:** the default path executes the checkpoint's own `configuration_molformer.py` and `modeling_molformer.py` after verifying their SHA-256 against the inline manifest. Section 3 explains why that is unavoidable for this checkpoint.
- **Runtime pin:** this repository pins `transformers==5.17.0`, not the fleet's 4.57.6, because the pinned upstream code calls an API that exists only in Transformers 5.5.0 and later.
- **Data contract:** records are `{{id, smiles, label}}`; SMILES are non-empty strings over the accepted character set, at most 200 tokens (`MAX_TOKENS`, the checkpoint's 202 position embeddings minus `<bos>`/`<eos>`), with unique ids and unique SMILES; at least 8 records and 3 per class, 2..20 classes. BYOD accepts CSV, JSON array or JSONL.
- **Validation is syntactic, not chemical:** this repository ships no cheminformatics toolkit, so it checks characters, bracket balance and ring-digit pairing — not valence or chemical plausibility. Use RDKit before trusting a SMILES set.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an unpublished or third-party proprietary structure is exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `ibm-research/MoLFormer-XL-both-10pct` snapshot (~187 MB in total) at revision `361063d0ad52…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `safetensors` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==5.17.0',
    'huggingface-hub==1.32.0',
    'tokenizers==0.23.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'molformer-chemistry-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/molformer_chemistry_pipeline/pipeline.py',
    'embedded_modules': ['src/molformer_chemistry_pipeline/pipeline.py', 'src/molformer_chemistry_pipeline/samples.py', 'src/molformer_chemistry_pipeline/metrics.py'],
    'module_sha256': 'd7af735beee8ece45443fecede0519c0d912f5c86e48539c7d4c4ecc87794ea5',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/molformer_chemistry_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/molformer_chemistry_pipeline/pipeline.py`

In [ ]:
"""MoLFormer-XL (`ibm-research/MoLFormer-XL-both-10pct`) DIMER pipeline: verified snapshot, SMILES
embeddings, and bounded molecular-property classification fine-tuning with a portable adapter.

Two things about this checkpoint shape the module and are stated rather than hidden:

* **It requires remote code.** `config.json` declares `model_type: "molformer"`, for which the
  installed Transformers has no native implementation, and the architecture is a linear-attention
  transformer that the generic classes do not implement. The loader therefore passes
  `trust_remote_code=True` — but only after `verify_snapshot` has checked the SHA-256 of the two
  Python files it will import, which are manifest entries for exactly that reason.
* **Its pinned code needs a recent Transformers.** The upstream revision pinned here calls
  `transformers.masking_utils.create_bidirectional_mask(inputs_embeds=...)`, which exists only in
  Transformers 5.5.0 and later, so this package pins `transformers==5.17.0` rather than the fleet's
  4.57.6. See `docs/WEIGHTS.md`.
* **Its attention is stochastic unless told otherwise.** MoLFormer uses linear attention with
  random feature maps. The upstream config ships `deterministic_eval: false`, which redraws those
  features on *every* forward pass, so two identical calls return different embeddings (measured:
  ~1e-3 on a pooled vector). This package loads with `deterministic_eval=True`, which keeps the
  feature weights the checkpoint itself stores, and exports those buffers with any adapter because
  they are serving state, not incidental.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` or `transformers` are imported (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "ibm-research/MoLFormer-XL-both-10pct"
MODEL_REVISION = "361063d0ad524ef77cf39b08469f6be770dc550f"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "molformer-xl-both-10pct"
ARTIFACT_FORMAT = "org.valcorza.molformer-chemistry.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
# The two files the loader executes. They are manifest entries, so their bytes are digest-verified
# before `trust_remote_code=True` imports them (MODEL_ASSET_SPEC RC3).
REMOTE_CODE_FILES = ("configuration_molformer.py", "modeling_molformer.py")
# Linear-attention random-feature buffers. They live in the checkpoint, they change during training,
# and inference depends on them, so an exported adapter carries them (ART3).
FEATURE_MAP_SUFFIX = ".feature_map.weight"
# Upstream defaults to redrawing the random features on every forward pass (`deterministic_eval:
# false` in config.json). DIMER needs a reproducible answer to the same question, so this package
# overrides that to True, which uses the feature weights stored in the pinned checkpoint.
DETERMINISTIC_EVAL = True

# Ceilings. config.json declares max_position_embeddings = 202, and the tokenizer wraps every SMILES
# in <bos> ... <eos>, so 200 SMILES tokens is the most a molecule can carry. Longer input is refused
# rather than truncated: a truncated SMILES is a different molecule, not a shorter one.
MAX_POSITION_EMBEDDINGS = 202
MAX_TOKENS = MAX_POSITION_EMBEDDINGS - 2
MAX_MOLECULES_PER_CALL = 64
HIDDEN_SIZE = 768  # config.json hidden_size; the width of every `embed` row
VOCAB_SIZE = 2362  # config.json vocab_size; equals the tokenizer's vocabulary
# A syntactic character set for SMILES, not a chemical validity check: this package has no cheminformatics
# toolkit, so it checks characters, bracket balance and ring-digit pairing and says so. Use RDKit to
# establish that a string is a real molecule.
SMILES_CHARS = frozenset(
    "BCNOPSFIHbcnopsfi"  # organic subset, aromatic lowercase, explicit H in brackets
    "lr"  # the second letters of Cl and Br
    "eaigAKLMTZ"  # letters that appear inside bracket atoms (Se, Na, Si, Mg, ...)
    "0123456789"  # ring-closure digits
    "()[]"  # branches and bracket atoms
    "=#$:/\\"  # bond symbols
    "+-@"  # charges and stereochemistry
    "%."  # two-digit ring closures and disconnected components
)


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    missing_code = [name for name in REMOTE_CODE_FILES if name not in listed]
    if missing_code:
        raise ValueError(
            f"manifest does not list the remote-code files {missing_code}; refusing to proceed, because "
            "the loader executes them and their digests must be verified first"
        )
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest, including the two executable Python files."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights and the upstream Python). `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1..MAX_MOLECULES_PER_CALL molecules as SMILES strings",
    "molecules": [1, MAX_MOLECULES_PER_CALL],
    "tokens_per_molecule": [1, MAX_TOKENS],
    "validation": (
        "syntactic only: character set, balanced parentheses and brackets, paired ring-closure digits, "
        "and the tokenized length ceiling. This package ships no cheminformatics toolkit and does not "
        "check valence, aromaticity or chemical plausibility -- use RDKit for that"
    ),
    "preprocessing": (
        "the pinned tokenizer splits SMILES into atom and symbol tokens (two-letter atoms such as Cl and "
        "Br and bracket atoms such as [C@H] stay single tokens), wraps them in <bos> ... <eos>, and pads "
        "to the longest molecule in the batch; embeddings are the mean of the last hidden state over "
        "non-padding tokens"
    ),
}


def check_smiles_syntax(smiles: str) -> list[str]:
    """Return a list of syntactic findings for one SMILES string; empty means it passed the checks.

    This is a string-level check, not a chemistry check: it catches typos and truncation, not
    impossible molecules.
    """
    findings: list[str] = []
    bad = sorted({ch for ch in smiles if ch not in SMILES_CHARS})
    if bad:
        findings.append(f"characters outside the accepted SMILES set: {bad!r}")
    for opener, closer, label in (("(", ")", "parenthesis"), ("[", "]", "bracket")):
        depth = 0
        for ch in smiles:
            depth += ch == opener
            depth -= ch == closer
            if depth < 0:
                findings.append(f"closing {label} before an opening one")
                break
        if depth > 0:
            findings.append(f"{depth} unclosed {label}(s)")
    digits: dict[str, int] = {}
    in_bracket = False
    for ch in smiles:
        if ch == "[":
            in_bracket = True
        elif ch == "]":
            in_bracket = False
        elif ch.isdigit() and not in_bracket:
            digits[ch] = digits.get(ch, 0) + 1
    odd = sorted(d for d, count in digits.items() if count % 2)
    if odd:
        findings.append(f"unpaired ring-closure digit(s): {odd}")
    return findings


def _check_molecules(molecules: Any, names: Any = None) -> tuple[list[str], list[str]]:
    """Raise TypeError/ValueError naming the first violated rule; return (smiles, ids).

    ``embed``, ``classify`` and ``validate_inputs`` all route through this function so their
    acceptance criteria cannot diverge.
    """
    if isinstance(molecules, str | bytes) or not isinstance(molecules, Sequence):
        raise TypeError("molecules must be a list of SMILES strings")
    if not 1 <= len(molecules) <= MAX_MOLECULES_PER_CALL:
        raise ValueError(f"molecules must hold 1..{MAX_MOLECULES_PER_CALL} items, got {len(molecules)}")
    checked: list[str] = []
    for i, smiles in enumerate(molecules):
        if not isinstance(smiles, str):
            raise TypeError(f"molecules[{i}] must be str, got {type(smiles).__name__}")
        if not smiles.strip():
            raise ValueError(f"molecules[{i}] is empty")
        if smiles != smiles.strip():
            raise ValueError(f"molecules[{i}] has leading or trailing whitespace; strip it first")
        findings = check_smiles_syntax(smiles)
        if findings:
            raise ValueError(
                f"molecules[{i}] ({smiles!r}) failed SMILES syntax checks: {'; '.join(findings)}"
            )
        checked.append(smiles)
    if names is None:
        ids = [f"mol-{i}" for i in range(len(checked))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(checked):
            raise ValueError("names must be a list with exactly one id per molecule")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return checked, ids


def _softmax(logits: Sequence[float]) -> list[float]:
    import math

    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


@dataclass
class MolformerPipeline:
    """MoLFormer-XL pipeline: `embed` always; `classify` after `adapt` or `from_artifact`."""

    _embedder: Callable[[list[str]], list[list[float]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[str]], list[list[float]]] | None = None
    _token_counter: Callable[[str], int] | None = None
    model: Any = None
    tokenizer: Any = None
    classifier_model: Any = None
    weights_dir: Path | None = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MolformerPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage and verify — including the two Python files — before importing model libraries and
        # before any remote code is executed (RTM-001, MODEL_ASSET_SPEC RC3).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        import torch
        from transformers import AutoModel, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            # The tokenizer is a native PreTrainedTokenizerFast: no remote code for this half.
            tokenizer = AutoTokenizer.from_pretrained(
                str(root), local_files_only=True, trust_remote_code=False
            )
            model = AutoModel.from_pretrained(
                str(root),
                local_files_only=True,
                trust_remote_code=True,
                deterministic_eval=DETERMINISTIC_EVAL,
            )
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(cls._make_embedder(model, tokenizer, resolved_device), resolved_device, messages)
        pipe.model, pipe.tokenizer, pipe.weights_dir = model, tokenizer, root
        pipe._token_counter = lambda smiles: len(tokenizer(smiles)["input_ids"])
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @staticmethod
    def _encode(tokenizer: Any, molecules: list[str], device: str) -> dict[str, Any]:
        batch = tokenizer(molecules, return_tensors="pt", padding=True)
        return {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask")}

    @classmethod
    def _make_embedder(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def embedder(molecules: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, molecules, device)
            # no_grad, not inference_mode: tensors produced here must stay usable by a later
            # training epoch that shares this module.
            with torch.no_grad():
                hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
            return pooled.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def classifier(molecules: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, molecules, device)
            with torch.no_grad():
                logits = model(**batch).logits
            return logits.float().cpu().tolist()

        return classifier

    def token_count(self, smiles: str) -> int:
        """Tokenized length including `<bos>`/`<eos>`; the ceiling applies to this, not to characters."""
        if self._token_counter is None:
            raise RuntimeError("token_count requires a pipeline built by from_pretrained")
        return self._token_counter(smiles)

    def _check_lengths(self, molecules: Sequence[str]) -> list[int]:
        counts = []
        for i, smiles in enumerate(molecules):
            n_tokens = self.token_count(smiles) - 2  # exclude <bos>/<eos>
            if n_tokens > MAX_TOKENS:
                raise ValueError(
                    f"molecules[{i}] tokenizes to {n_tokens} tokens; the ceiling is {MAX_TOKENS} "
                    f"(max_position_embeddings {MAX_POSITION_EMBEDDINGS} minus <bos> and <eos>). "
                    "A truncated SMILES is a different molecule, so this is refused rather than cut."
                )
            counts.append(n_tokens)
        return counts

    # -- public stages ----------------------------------------------------------------------------

    def embed(self, molecules: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Mean-pooled last-hidden-state representation per molecule (HIDDEN_SIZE floats each)."""
        checked, ids = _check_molecules(molecules, names)
        token_counts = self._check_lengths(checked)
        vectors = self._embedder(checked)
        if len(vectors) != len(checked) or any(len(v) != HIDDEN_SIZE for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": HIDDEN_SIZE,
            "pooling": "mean of the last hidden state over non-padding tokens",
            "unit": "one vector per molecule; representations, not property predictions",
            "tokens": token_counts,
            "n_molecules": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def classify(self, molecules: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Class scores and argmax label per molecule; requires a prior `adapt` or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        checked, ids = _check_molecules(molecules, names)
        token_counts = self._check_lengths(checked)
        logits = self._classifier(checked)
        predictions = []
        for mid, smiles, n_tokens, row in zip(ids, checked, token_counts, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": mid,
                    "smiles": smiles,
                    "tokens": n_tokens,
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_molecules": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out property-classification metrics (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, classes=self.classes)
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_MOLECULES_PER_CALL):
            chunk = records[start : start + MAX_MOLECULES_PER_CALL]
            result = self.classify([r["smiles"] for r in chunk], names=[r["id"] for r in chunk])
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        epochs: int = 4,
        learning_rate: float = 1e-4,
        batch_size: int = 8,
        trainable_layers: int = 2,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a property-classification head on the verified base.

        Builds `MolformerForSequenceClassification` from the pinned checkpoint (the head is newly
        initialised), freezes every parameter except the head and the last `trainable_layers`
        encoder layers, and runs AdamW for `epochs` passes. Validation records are monitored per
        epoch only; the final epoch's weights are kept (no selection).
        """
        if self.model is None or self.tokenizer is None or self.weights_dir is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_MOLECULES_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_MOLECULES_PER_CALL}")
        if not 0 <= int(trainable_layers) <= 12:
            raise ValueError("trainable_layers must be in 0..12 (the checkpoint has 12 encoder layers)")
        train_manifest = validate_dataset(train_records, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, classes=class_list)

        import random

        import torch
        from transformers import AutoModelForSequenceClassification

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        clf = AutoModelForSequenceClassification.from_pretrained(
            str(self.weights_dir),
            local_files_only=True,
            trust_remote_code=True,
            num_labels=len(class_list),
            deterministic_eval=DETERMINISTIC_EVAL,
        ).to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        layers = clf.molformer.encoder.layer
        for layer in layers[len(layers) - int(trainable_layers) :] if trainable_layers else []:
            for p in layer.parameters():
                p.requires_grad = True
        for p in clf.classifier.parameters():
            p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        examples = [(r["smiles"], label_index[r["label"]]) for r in train_records]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)

        history: list[dict[str, Any]] = []
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(examples)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                rows = [examples[i] for i in order[start : start + int(batch_size)]]
                batch = self._encode(self.tokenizer, [s for s, _ in rows], self.device)
                labels = torch.tensor([y for _, y in rows], device=self.device)
                optimizer.zero_grad()
                out = clf(**batch, labels=labels)
                out.loss.backward()
                optimizer.step()
                total_loss += float(out.loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of the classification head"
            + (f" and the last {int(trainable_layers)} encoder layer(s)" if trainable_layers else ""),
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_layers": int(trainable_layers),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        state = self.classifier_model.state_dict()
        # Trained parameters plus the linear-attention feature buffers: training redraws those
        # buffers, and inference depends on them, so an adapter without them does not reproduce
        # the adapted model (ART3).
        serving_state = sorted(k for k in state if k.endswith(FEATURE_MAP_SUFFIX))
        tensors = {
            k: v.detach().cpu().contiguous() for k, v in state.items() if k in names or k in serving_state
        }
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "requires_remote_code": True,
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "serving_state_tensors": serving_state,
            "deterministic_eval": DETERMINISTIC_EVAL,
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(f"artifact was trained on {base}, this package pins {MODEL_ID}@{MODEL_REVISION}")
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file
        from transformers import AutoModelForSequenceClassification

        clf = AutoModelForSequenceClassification.from_pretrained(
            str(self.weights_dir),
            local_files_only=True,
            trust_remote_code=True,
            num_labels=len(classes),
            deterministic_eval=DETERMINISTIC_EVAL,
        )
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        if set(tensors) != set(manifest.get("tensors", [])):
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        _missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MolformerPipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def validate_inputs(
    molecules: Sequence[str],
    *,
    names: Sequence[str] | None = None,
    token_counter: Callable[[str], int] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, verdict).

    `token_counter` is the pipeline's `token_count`; without it the manifest reports character
    lengths and says that the token ceiling was not checked, rather than guessing at it.
    """
    checked, ids = _check_molecules(molecules, names)
    rows = []
    for mid, smiles in zip(ids, checked, strict=True):
        row: dict[str, Any] = {"id": mid, "smiles": smiles, "characters": len(smiles)}
        if token_counter is not None:
            n_tokens = token_counter(smiles) - 2
            if n_tokens > MAX_TOKENS:
                raise ValueError(
                    f"{mid} tokenizes to {n_tokens} tokens; the ceiling is {MAX_TOKENS} "
                    f"(max_position_embeddings {MAX_POSITION_EMBEDDINGS} minus <bos> and <eos>)"
                )
            row["tokens"] = n_tokens
        rows.append(row)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": rows,
        "n_molecules": len(checked),
        "token_ceiling_checked": token_counter is not None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "requires_remote_code": True,
    }

**Module 2/3:** `src/molformer_chemistry_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample molecules and the labelled-dataset contract for property classification.

The tutorial task is synthetic but chemically literal: each pair of molecules is a pair of
**constitutional isomers** — the same molecular formula, differing only in connectivity. One member
is an unbranched primary alcohol (`CCCCCCCO`); the other carries the same atoms with one methyl
branch moved onto the chain (`CCCC(C)CCO`). Molecular formula therefore carries no signal by
construction, so a composition baseline sits at chance and any separation the model achieves comes
from reading structure. This is sanity evidence for the adaptation contract, not a chemistry
benchmark, and the classes are a generator rule rather than a measured property (NOTEBOOK_SPEC 2.0
DAT8).

One honest caveat, stated here and in the tutorial: a branched SMILES contains `(` and `)`, so a
*character-level* baseline that counted punctuation could separate these classes trivially. The
baseline this package ships counts heavy atoms — the molecular formula — because that is the
chemically meaningful confounder to rule out.
"""

from __future__ import annotations

import csv
import hashlib
import json
import random
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TOKENS, check_smiles_syntax` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.chemistry.smiles-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = ("branched", "linear")
SAMPLE_SEED = 20260918
SAMPLE_SIZE = 64  # 32 pairs
MIN_CARBONS = 6
MAX_CARBONS = 14
# Terminal groups, so that pairs differ from one another rather than only in chain length. Both
# members of a pair carry the same terminus, so the terminus can never separate the two classes.
TERMINI: tuple[str, ...] = ("O", "N", "S", "F", "Cl", "Br", "I")
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_CLASSES = 20
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "smiles", "label")
_ATOM = re.compile(r"Cl|Br|[BCNOPSFI]|\[[^\]]*\]|[bcnops]")


def molecular_formula(smiles: str) -> dict[str, int]:
    """Heavy-atom counts for the organic subset this sample uses (no implicit hydrogens).

    This is a string-level count, not a cheminformatics parse: it recognises the two-letter atoms
    `Cl` and `Br`, single-letter organic-subset atoms, aromatic lowercase atoms, and bracket atoms
    as one atom each. It exists to give the tutorial a composition baseline, not to replace RDKit.
    """
    counts: dict[str, int] = {}
    for token in _ATOM.findall(smiles):
        if token.startswith("["):
            inner = token[1:-1]
            match = re.search(r"[A-Z][a-z]?|[bcnops]", inner)
            symbol = match.group(0) if match else inner
        else:
            symbol = token
        symbol = symbol.capitalize() if len(symbol) > 1 else symbol.upper()
        counts[symbol] = counts.get(symbol, 0) + 1
    return counts


def formula_string(smiles: str) -> str:
    """`molecular_formula` rendered in a stable order, e.g. `C7O1`."""
    counts = molecular_formula(smiles)
    return "".join(f"{symbol}{counts[symbol]}" for symbol in sorted(counts))


def branch_positions(n_carbons: int) -> list[int]:
    """Positions at which a methyl branch can sit without recreating the linear chain."""
    return list(range(2, n_carbons - 2))


def make_isomer_pair(n_carbons: int, branch_at: int, terminus: str = "O") -> tuple[str, str]:
    """A linear molecule and a branched constitutional isomer of it: same formula, different skeleton.

    `CCCCCCCO` and `CCCC(C)CCO` both have `n_carbons` carbons and one oxygen: the branched member
    spends one of its carbons as a methyl substituent instead of extending the chain. `terminus` is
    the group at the end of the chain and is identical for both members, so it cannot carry the label.
    """
    if not MIN_CARBONS <= n_carbons <= MAX_CARBONS:
        raise ValueError(f"n_carbons must be in {MIN_CARBONS}..{MAX_CARBONS}, got {n_carbons}")
    if branch_at not in branch_positions(n_carbons):
        raise ValueError(f"branch_at must be one of {branch_positions(n_carbons)} for {n_carbons} carbons")
    if terminus not in TERMINI:
        raise ValueError(f"terminus must be one of {list(TERMINI)}, got {terminus!r}")
    linear = "C" * n_carbons + terminus
    chain = n_carbons - 1  # one carbon becomes the branch
    branched = "C" * branch_at + "(C)" + "C" * (chain - branch_at) + terminus
    return linear, branched


def generate_sample_dataset(seed: int = SAMPLE_SEED, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """`size` labelled molecules (half `linear`, half `branched`), deterministic for a given seed.

    Every branched molecule is a constitutional isomer of its linear partner: identical formula,
    different connectivity.
    """
    if size < 2 or size % 2:
        raise ValueError("size must be an even number >= 2 (one linear molecule per branched isomer)")
    skeletons = [(n, terminus) for terminus in TERMINI for n in range(MIN_CARBONS, MAX_CARBONS + 1)]
    if size // 2 > len(skeletons):
        raise ValueError(
            f"size {size} needs {size // 2} distinct (chain length, terminus) pairs but only "
            f"{len(skeletons)} exist; widen MIN_CARBONS..MAX_CARBONS or TERMINI"
        )
    rng = random.Random(seed)
    rng.shuffle(skeletons)
    records: list[dict[str, Any]] = []
    for i, (n_carbons, terminus) in enumerate(skeletons[: size // 2]):
        branch_at = rng.choice(branch_positions(n_carbons))
        linear, branched = make_isomer_pair(n_carbons, branch_at, terminus)
        records.append({"id": f"lin-{i:03d}", "smiles": linear, "label": "linear"})
        records.append({"id": f"bra-{i:03d}", "smiles": branched, "label": "branched"})
    return records


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, smiles, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps([[r["id"], r["smiles"], r["label"]] for r in records], separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled molecule dataset against the contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). SMILES are checked
    syntactically only — character set, bracket balance, ring-digit pairing — because this package
    ships no cheminformatics toolkit; the manifest says so rather than implying chemical validation.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'smiles', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    seen_smiles: dict[str, str] = {}
    counts_by_class: dict[str, int] = {}
    lengths: list[int] = []
    formulas: dict[str, set[str]] = {}
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        smiles = rec["smiles"]
        if not isinstance(smiles, str) or not smiles.strip():
            raise ValueError(f"record[{i}] ({rid}) smiles must be a non-empty string")
        if smiles != smiles.strip():
            raise ValueError(f"record[{i}] ({rid}) smiles has leading or trailing whitespace")
        findings = check_smiles_syntax(smiles)
        if findings:
            raise ValueError(f"record[{i}] ({rid}) failed SMILES syntax checks: {'; '.join(findings)}")
        if smiles in seen_smiles:
            raise ValueError(f"record[{i}] ({rid}) duplicates the SMILES of {seen_smiles[smiles]!r}")
        seen_smiles[smiles] = rid
        lengths.append(len(smiles))
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts_by_class[label] = counts_by_class.get(label, 0) + 1
        formulas.setdefault(formula_string(smiles), set()).add(label)
    if classes is None:
        class_list = sorted(counts_by_class)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts_by_class) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts_by_class.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    shared = sorted(f for f, labels in formulas.items() if len(labels) > 1)
    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "validation": "syntactic SMILES checks only; no cheminformatics toolkit is used",
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts_by_class.get(c, 0) for c in class_list},
        "smiles_characters": {
            "min": min(lengths),
            "max": max(lengths),
            "mean": round(sum(lengths) / len(lengths), 1),
        },
        "distinct_formulas": len(formulas),
        "formulas_shared_across_classes": len(shared),
        "ceilings": {
            "max_tokens": MAX_TOKENS,
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes independent molecules, SPL3).

    Real chemical datasets are rarely independent: analogues from one series share a scaffold, and a
    random split lets the model memorise it. A scaffold or cluster split is the right tool there;
    the tutorial's molecules are generated independently, which is why a random split is honest here.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied dataset (CSV with `id,smiles,label`, JSON array, or JSONL).

    SMILES are stripped of surrounding whitespace only; nothing else is rewritten (VAL7). The
    records are then validated with `validate_dataset`, whose errors name the offending row.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        raise ValueError(f"BYOD dataset file is empty: {path}")
    suffix = path.suffix.lower()
    records: list[dict[str, Any]] = []
    if suffix == ".csv":
        reader = csv.DictReader(text.splitlines())
        header = [h.strip() for h in (reader.fieldnames or [])]
        missing = [c for c in REQUIRED_COLUMNS if c not in header]
        if missing:
            raise ValueError(f"CSV header {header} is missing required column(s) {missing}")
        for row in reader:
            records.append({c: (row.get(c) or "").strip() for c in REQUIRED_COLUMNS})
    elif suffix == ".jsonl":
        for line_no, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
    elif suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is not valid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError("JSON dataset must be a top-level array of objects")
        records = data
    else:
        raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json or .jsonl")
    for rec in records:
        if isinstance(rec, Mapping) and isinstance(rec.get("smiles"), str):
            rec["smiles"] = rec["smiles"].strip()
    validate_dataset(records)
    return [dict(r) for r in records]


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records as the BYOD CSV shape (`id,smiles,label`), so users have a template."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(REQUIRED_COLUMNS)
        for r in records:
            writer.writerow([r["id"], r["smiles"], r["label"]])
    return out

**Module 3/3:** `src/molformer_chemistry_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Classification metrics and trivial baselines for MoLFormer property-classification adaptation.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .samples import formula_string` removed — names are kernel globals defined by the carried modules


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def formula_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict from the molecular formula alone, learned on the training split only (SPL8).

    Molecular formula is the confounder worth ruling out in any molecular-property task: if the
    formula already separates the classes, the model has not had to learn chemistry. The rule is a
    lookup -- for each formula seen in training, predict its majority class; formulas never seen in
    training fall back to the overall training majority. On the tutorial sample every formula appears
    in both classes by construction (the pairs are constitutional isomers), so this baseline is
    expected to sit at chance.
    """
    class_list = list(classes)
    by_formula: dict[str, dict[str, int]] = {}
    overall: dict[str, int] = {}
    for record in train_records:
        formula = formula_string(record["smiles"])
        by_formula.setdefault(formula, {})
        by_formula[formula][record["label"]] = by_formula[formula].get(record["label"], 0) + 1
        overall[record["label"]] = overall.get(record["label"], 0) + 1
    fallback = max(sorted(overall), key=overall.__getitem__)
    lookup = {formula: max(sorted(counts), key=counts.__getitem__) for formula, counts in by_formula.items()}
    predicted, unseen = [], 0
    for record in eval_records:
        formula = formula_string(record["smiles"])
        if formula in lookup:
            predicted.append(lookup[formula])
        else:
            predicted.append(fallback)
            unseen += 1
    metrics = classification_metrics([r["label"] for r in eval_records], predicted, None, class_list)
    return {
        "baseline": "molecular-formula lookup",
        "distinct_train_formulas": len(lookup),
        "ambiguous_train_formulas": sum(1 for counts in by_formula.values() if len(counts) > 1),
        "eval_formulas_unseen_in_train": unseen,
        "fallback_label": fallback,
        **metrics,
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `361063d0ad52…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MolformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "molformer-xl-both-10pct",
  "modelId": "ibm-research/MoLFormer-XL-both-10pct",
  "revision": "361063d0ad524ef77cf39b08469f6be770dc550f",
  "files": [
    {
      "path": "README.md",
      "bytes": 6279,
      "sha256": "fff331c6a973fa6995662ac31354c702ac0f4805d494e7506b979a3a6cee6f44"
    },
    {
      "path": "config.json",
      "bytes": 1015,
      "sha256": "3ef9eaac8c7ca6282fd6256ed038d151bd4ff42a4ff855367e0d7197bbc1c284"
    },
    {
      "path": "configuration_molformer.py",
      "bytes": 7101,
      "sha256": "b88ea8d4b7b5e54f4f186cc7a230eff308928020a030f153a038fd12c05e3bed"
    },
    {
      "path": "model.safetensors",
      "bytes": 187248784,
      "sha256": "0795977fe7192c4acdaf052f0e8464af57bc4bb59211271c5e61aaba2637b9c6"
    },
    {
      "path": "modeling_molformer.py",
      "bytes": 36884,
      "sha256": "6f1ef72022de2c69e95661899422a7bb39a40a2cc5a6cb6216f14e9b7d84559c"
    },
    {
      "path": "tokenizer.json",
      "bytes": 54010,
      "sha256": "3df1f2219653c44fac9fa03b7f788b372eb2544ecc176737bb9aca8411b471a5"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1132,
      "sha256": "56f93dc43e4383fcfc6342e09d782ef38d48ef227215a4a92ff8762fd5b8ff3e"
    }
  ],
  "totalBytes": 187355205
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MolformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample molecules, validation and split

The default dataset is generated in code with a fixed seed: 32 pairs, each a linear molecule and a branched **constitutional isomer** of it. Both members have the same molecular formula and the same terminal group; they differ only in where one carbon sits. `validate_dataset` checks the schema, the SMILES syntax, duplicate ids and SMILES, and class coverage before any model runs, and reports how many distinct formulas there are and how many of them appear in **both** classes — on the sample, all of them, which is the property the whole comparison rests on. `split_dataset` shuffles within each class and cuts 20 % validation / 25 % test.

Look for: 64 molecules, classes `['branched', 'linear']`, 32 distinct formulas all shared across classes, splits 36/12/16, and a written `outputs/molformer_chemistry_sample_dataset.csv` — the file shape BYOD expects. One honest caveat printed with them: a branched SMILES contains `(` and `)`, so a *character-level* baseline could separate these classes trivially; the baseline this notebook ships counts atoms, because molecular formula is the chemically meaningful confounder to rule out.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    records = generate_sample_dataset()
    data_source = f'synthetic constitutional-isomer dataset (seed {SAMPLE_SEED}, {SAMPLE_SIZE} molecules)'

dataset_manifest = validate_dataset(records)
CLASSES = dataset_manifest['classes']
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(records, 'outputs/molformer_chemistry_sample_dataset.csv')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'distinct_formulas': dataset_manifest['distinct_formulas'], 'formulas_shared_across_classes': dataset_manifest['formulas_shared_across_classes'], 'validation': dataset_manifest['validation']})
print({'smiles_characters': dataset_manifest['smiles_characters'], 'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})

if not USE_BYOD:
    example_linear = next(r for r in records if r['label'] == 'linear')
    example_branched = next(r for r in records if r['id'].endswith(example_linear['id'][-3:]) and r['label'] == 'branched')
    print({'pair': (example_linear['smiles'], example_branched['smiles']), 'formula': (formula_string(example_linear['smiles']), formula_string(example_branched['smiles']))})
    assert formula_string(example_linear['smiles']) == formula_string(example_branched['smiles'])
    print('caveat: the branched SMILES contains ( and ), so a character-level baseline would separate these classes trivially; the formula baseline below deliberately counts atoms instead')

## 5. How a molecule becomes tokens

The tokenizer is a native `PreTrainedTokenizerFast` reading the checkpoint's own `tokenizer.json` — no remote code for this half. It splits SMILES into chemical tokens rather than characters: two-letter atoms such as `Cl` and `Br` stay one token, bracket atoms such as `[C@H]` stay one token, and branch parentheses and bond symbols are tokens of their own.

The ceiling that follows from the checkpoint is `MAX_TOKENS = 200` (its 202 position embeddings minus `<bos>` and `<eos>`). A molecule past that is **refused rather than truncated** — a truncated SMILES is a different molecule, not a shorter one — and the cell demonstrates that refusal along with three syntax rejections.

In [ ]:
for smiles in ['CCCCCCCO', 'CCCC(C)CCO', 'ClCCBr', 'C[C@H](N)C(=O)O', 'c1ccccc1O']:
    print({'smiles': smiles, 'tokens': pipe.token_count(smiles), 'formula': formula_string(smiles)})

print({'max_tokens': MAX_TOKENS, 'max_position_embeddings': MAX_POSITION_EMBEDDINGS, 'max_molecules_per_call': MAX_MOLECULES_PER_CALL})

for probe in ['CCCC(CCO', 'CC*C', 'c1ccccO', ' CCO']:
    try:
        validate_inputs([probe])
        print({'probe': probe, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': probe, 'rejected': str(exc)[:100]})
try:
    pipe.embed(['C' * (MAX_TOKENS + 1)])
except ValueError as exc:
    print({'too_long': str(exc)[:130]})

input_manifest = validate_inputs([r['smiles'] for r in test_records[:4]], names=[r['id'] for r in test_records[:4]], token_counter=pipe.token_count)
print({'verdict': input_manifest['verdict'], 'n_molecules': input_manifest['n_molecules'], 'token_ceiling_checked': input_manifest['token_ceiling_checked'], 'requires_remote_code': input_manifest['requires_remote_code']})

## 6. Molecule embeddings (representation, not prediction)

`pipe.embed` runs the verified encoder and returns one 768-dimensional vector per molecule: the mean of the last hidden state over non-padding tokens. Embeddings are representations — they carry no label and no metric of their own; a downstream labelled task is what gives them meaning (EVAL9). The cell embeds eight validation molecules, writes them with their ids to `outputs/molformer_chemistry_embeddings.csv` (OUT4), and prints the mean cosine similarity within and between classes as an inspection, not an evaluation.

Because this package loads with `deterministic_eval=True`, running this cell twice gives exactly the same vectors. Under the upstream default it would not — see Section 3.

In [ ]:
import csv
import math

embed_records = val_records[:8]
embedding_result = pipe.embed([r['smiles'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_molecules': embedding_result['n_molecules'], 'dimension': embedding_result['dimension'], 'tokens': embedding_result['tokens'], 'pooling': embedding_result['pooling']})

repeat = pipe.embed([embed_records[0]['smiles']])['embeddings'][0]
print({'deterministic_eval': DETERMINISTIC_EVAL, 'same_vector_on_a_second_call': repeat == vectors[0]})

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

within, between = [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        sim = cosine(vectors[i], vectors[j])
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(sim)
print({'mean_cosine_within_class': round(sum(within) / len(within), 4) if within else None, 'mean_cosine_between_classes': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/molformer_chemistry_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'smiles', 'label'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, vec in zip(embed_records, vectors):
        writer.writerow([r['id'], r['smiles'], r['label']] + [f'{x:.6f}' for x in vec])
print('wrote outputs/molformer_chemistry_embeddings.csv')

## 7. Baselines on the test split

Two trivial predictors set the floor before any training (EVAL10/EVAL11). `majority_baseline` predicts the most frequent training class — 0.5 on a balanced split. `formula_baseline` looks up each test molecule's **molecular formula** among the training formulas and predicts that formula's majority training class, falling back to the overall training majority for formulas it has not seen (SPL8: the lookup is built on the training split only).

On this sample the formula baseline is structurally helpless, and the printout says why: every formula occurs exactly twice — once as a linear molecule, once as its branched isomer — so a formula seen in training is a coin flip, and a formula that was split across train and test is unseen and falls back. That is the control working as intended, not a bug. On your own data, read this baseline first: if the formula already predicts your label, the model does not have to learn any chemistry to score well.

In [ ]:
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
baseline_formula = formula_baseline(train_records, test_records, CLASSES)
print({k: baseline_formula[k] for k in ('baseline', 'distinct_train_formulas', 'ambiguous_train_formulas', 'eval_formulas_unseen_in_train', 'accuracy', 'macro_f1')})

## 8. Bounded fine-tuning

`pipe.adapt` builds `MolformerForSequenceClassification` from the verified checkpoint (the head is newly initialised — the load report says so), freezes every parameter except the head and the last `TRAINABLE_LAYERS` encoder layers, and runs AdamW with the hyperparameters below (FT4/FT6): tutorial values chosen for a few seconds of CPU, not production settings. Validation metrics are computed after each epoch for **monitoring only**; the final epoch's weights are kept, so no selection happens on the validation split (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 9 is where quality is measured.

One MoLFormer-specific detail: during training the model is in `train()` mode, so its linear-attention random features **are** redrawn each step regardless of `deterministic_eval`. That is upstream's intended stochastic regularisation; it also means the feature buffers at the end of training differ from the checkpoint's, which is why Section 10 exports them with the adapter.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_layers=TRAINABLE_LAYERS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 2)
print({'method': adapt_result['method'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print(step)

## 9. Held-out evaluation

`pipe.evaluate` classifies every molecule of a split and reports `accuracy`, `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a class), per-class precision/recall/F1 with support, and `auroc` (ranking quality of the positive-class score, independent of the argmax threshold). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7). These are tutorial metrics on a synthetic 16-molecule split (EVAL6): one holdout, no dispersion estimate. The report, with both baselines and the deltas against them, is written to `outputs/molformer_chemistry_evaluation_report.json`.

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'molecular-property classification (bounded fine-tuning of MoLFormer-XL)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout; not a chemistry benchmark',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'majority': baseline_majority, 'formula': baseline_formula},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/molformer_chemistry_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/molformer_chemistry_evaluation_report.json'})

## 10. Inference on new molecules, artifact export and fresh reload

`pipe.classify` returns, per molecule, the argmax `label`, its `score` and the full `scores` dictionary in class order. The scores are softmax outputs of a head trained on a few dozen molecules — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3).

`pipe.save_artifact` then writes the trained tensors **plus the 12 linear-attention feature buffers** as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the class order, the tensor names, which of them are serving state, the file size and SHA-256, and the adaptation configuration (OUT8/ART3). Those buffers are not an afterthought: training redraws them, inference depends on them, and an adapter without them does not reproduce the adapted model. `MolformerPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical labels and scores within `1e-5` (VER4).

In [ ]:
if USE_BYOD:
    new_records = test_records[:6]
    new_source = 'first six BYOD test-split molecules'
else:
    new_records = generate_sample_dataset(seed=7, size=6)
    new_source = 'freshly generated isomer pairs (seed 7)'
inference_result = pipe.classify([r['smiles'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'new_source': new_source, 'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'smiles': p['smiles'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on generated labels, not an evaluation'})

with open('outputs/molformer_chemistry_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'smiles', 'tokens', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['smiles'], p['tokens'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])

artifact_dir = Path('outputs/molformer_chemistry_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'requires_remote_code': artifact_manifest['requires_remote_code'], 'deterministic_eval': artifact_manifest['deterministic_eval'], 'n_tensors': len(artifact_manifest['tensors']), 'serving_state_tensors': len(artifact_manifest['serving_state_tensors']), 'files': artifact_manifest['files']})

reloaded_pipe = MolformerPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['smiles'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff})

## 11. Result export and provenance

The last output, `outputs/molformer_chemistry_result.json`, gathers what a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision and licence, **the fact that remote code was executed and which files were verified first**, the `deterministic_eval` override, the dataset source and digest, the adaptation configuration, baseline and held-out metrics, the new-molecule predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

result_payload = {
    'task': 'molecular-property classification adaptation (MoLFormer-XL)',
    'pipeline_class': 'MolformerPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'remote_code_executed': True,
    'remote_code_files': list(REMOTE_CODE_FILES),
    'deterministic_eval': DETERMINISTIC_EVAL,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'dataset_manifest': dataset_manifest,
    'embedding_summary': {'n_molecules': embedding_result['n_molecules'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling']},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'safetensors': safetensors.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/molformer_chemistry_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The fine-tuned head separates molecules that share a molecular formula and differ only in where one carbon sits — which the formula baseline cannot do, by construction. That is the claim: the model's representation carries structure, and a bounded adaptation can read it. The test split has 16 synthetic molecules, the metrics come from one seeded holdout with no dispersion estimate, and the classes are a generator rule rather than a measured property. So a perfect score says the adaptation contract works, not that MoLFormer predicts solubility, toxicity, binding affinity or any real endpoint.

Three things to carry to real data. **Splits:** analogues from one chemical series share a scaffold, so a random split lets the model memorise it — use a scaffold or cluster split and expect lower, truer numbers. **Validation:** this repository checks SMILES *syntax*, not chemistry; run RDKit over your set before you trust it, because a syntactically fine string can be a chemically impossible molecule and the pipeline will happily embed it. **Applicability domain:** the pretraining corpus is drug-like PubChem and ZINC chemistry, so inorganics, organometallics, polymers and very large molecules are out of distribution with no error to warn you — and the 200-token ceiling refuses them rather than truncating.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model **and the Python it executes**, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against trivial baselines on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or chemical validity.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 0` to train the head alone and compare; lower `EPOCHS` to 1 to see an under-trained head where AUROC may be high while accuracy sits near 0.5; or bring your own labelled set through BYOD and read the formula baseline first — if it already separates your classes, your labels may be predictable from composition alone.

## References

- Repository README: https://github.com/kurtvalcorza/molformer-chemistry-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/molformer-chemistry-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct
- Upstream code: https://github.com/IBM/molformer
- Ross, J., Belgodere, B., Chenthamarakshan, V., Padhi, I., Mroueh, Y., Das, P. (2021). Large-Scale Chemical Language Representations Capture Molecular Structure and Properties. arXiv:2106.09553. https://arxiv.org/abs/2106.09553